In [1]:
import sys
sys.path.append('../../1_figure_CL_proof_of_concept/code/')
import utils_00 as gf_utils
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as st

In [2]:
expected_mutations = pd.read_csv('../data/MPN_expected_mutations.csv')[['sample','name','mutation_freq']]
expected_mutations.rename(columns={'mutation_freq': 'expected_frequency_from_bulk'}, inplace=True)

expected_mutations['HGVSc'] = expected_mutations['name'].str.split(' ').str[1]
expected_mutations['gene'] = expected_mutations['name'].str.split(' ').str[0]

### drop mutations with 0 expected frequency (these were annotated by mistake in initial clinical reporting)
expected_mutations = expected_mutations.loc[expected_mutations['expected_frequency_from_bulk'] > 0]

In [3]:
dir = '../data/unexpected_gapfills/'

### load in tables of all gapfills seen in each library
merged_table_1 = gf_utils.make_merge_table(dir + 'unexpected_gapfills_1', label_control_column=True, lib='1', control_idx = '16')
merged_table_2 = gf_utils.make_merge_table(dir + 'unexpected_gapfills_2', label_control_column=True, lib='2', control_idx = '15')
merged_table_4plex = gf_utils.make_merge_table(dir + 'unexpected_gapfills_4plex', label_control_column=False, lib='4plex')
merged_table_1plex = gf_utils.make_merge_table(dir + 'unexpected_gapfills_1plex', label_control_column=False, lib='1plex')

In [4]:
merge_columns = ['gapfill', 'gapfill_from_transcriptome','gapfill_start','gap_probe_sequence','likelihood_given_wt_edit_dist','lhs_probe','rhs_probe']

# Merge only if the DataFrames are not empty
if not merged_table_2.empty:
    merged_table = merged_table_1.merge(merged_table_2, on=merge_columns, how='outer')
else:
    merged_table = merged_table_1.copy()

if not merged_table_4plex.empty:
    merged_table = merged_table.merge(merged_table_4plex, on=merge_columns, how='outer')

if not merged_table_1plex.empty:
    merged_table = merged_table.merge(merged_table_1plex, on=merge_columns, how='outer')
    
### manual fix for dual probes
merged_table.loc[(merged_table['lhs_probe'] == 'ATTTAGAGGATAAGGCGGCAGTAGT') & (merged_table['rhs_probe'] == 'TGTGTTCGCTGTAGATCTGACGTAC'),'rhs_probe'] = 'TGTGTTCGCTGTAGATCTGACGTAC/TGTGTAAGGCGGCAGTAGTTGTGTT'
merged_table.loc[(merged_table['lhs_probe'] == 'ATTTAGAGGATAAGGCGGCAGTAGT') & (merged_table['rhs_probe'] == 'TGTGTAAGGCGGCAGTAGTTGTGTT'),'rhs_probe'] = 'TGTGTTCGCTGTAGATCTGACGTAC/TGTGTAAGGCGGCAGTAGTTGTGTT'
merged_table.loc[(merged_table['rhs_probe'] == 'AATAGCTCATTAAAGTCATCTACCG') & (merged_table['lhs_probe'] == 'GCTGGCTGTCAGCGGGTACCTTGCC'),'lhs_probe'] = 'GCTGGCTGTCAGCGGGTACCTTGCC/GCTCTCTGTCAGCGGGTACCTTGCC'
merged_table.loc[(merged_table['rhs_probe'] == 'AATAGCTCATTAAAGTCATCTACCG') & (merged_table['lhs_probe'] == 'GCTCTCTGTCAGCGGGTACCTTGCC'),'lhs_probe'] = 'GCTGGCTGTCAGCGGGTACCTTGCC/GCTCTCTGTCAGCGGGTACCTTGCC'

merged_table = gf_utils.sum_probe_counts(merged_table)


In [5]:
## reformat table to long format - 1 row for each sample/gapfill/probe combination

id_vars = list(merge_columns) + list(merged_table.columns[merged_table.columns.str.contains('control')])
value_vars = merged_table.columns.drop(id_vars)

samples = value_vars.str.split('probe_idx_').str[1].dropna().unique()
print('samples to parse:', samples)

dfs = []
for sample in samples:
    df_tmp = merged_table[id_vars].copy()
    for col in value_vars:
        if col.endswith(sample):
            df_tmp[col.replace('_' + sample, '')] = merged_table[col]
    df_tmp['sample'] = sample
    dfs.append(df_tmp)

merged_long = pd.concat(dfs, ignore_index=True)

merged_long['original_name'] = merged_long['name'].copy()

### manual fix for extra added probes
merged_long['name'] = merged_long['name'].str.replace('_extra','')
### manual fix for TET2 c.3218G>A
merged_long.loc[merged_long['name'] == 'TET2 c.3218G>A','gapfill_start'] = '3217'

## drop rows where probe_idx is NaN
merged_long = merged_long.loc[merged_long['probe_idx'].notna()]

### add HGVSc based on observed gapfill
merged_long = gf_utils.name_variants_by_gapfill(merged_long, expected_mutations, merge_columns)

## reorder columns for better interpretability
cols = ['name','HGVSc'] + merge_columns + [col for col in merged_long.columns if col not in (['name','HGVSc'] + merge_columns)]
merged_long = merged_long[cols]

## manual fix for dual probes
merged_long.loc[(merged_long['name'] == 'ASXL1 c.2081_2099dup_0') & (merged_long['gapfill'] == '') & (merged_long['gapfill_start'].isna()),'HGVSc'] = 'ASXL1 c.2081_2099dup'
merged_long.loc[(merged_long['name'] == 'ASXL1 c.2081_2099dup_1') & (merged_long['gapfill'] == '') & (merged_long['gapfill_start'].notna()),'HGVSc'] = 'ASXL1 wildtype'
merged_long.loc[(merged_long['name'] == 'ASXL1 c.2081_2099dup_0') & (merged_long['gapfill'] == '') & (merged_long['gapfill_start'].isna()),'name'] = 'ASXL1 c.2081_2099dup'
merged_long.loc[(merged_long['name'] == 'ASXL1 c.2081_2099dup_1') & (merged_long['gapfill'] == '') & (merged_long['gapfill_start'].notna()),'name'] = 'ASXL1 c.2081_2099dup'

merged_long.loc[(merged_long['name'] == 'NFE2 c.782_785del_0') & (merged_long['gapfill'] == '') & (merged_long['gapfill_start'].isna()),'HGVSc'] = 'NFE2 c.782_785del'
merged_long.loc[(merged_long['name'] == 'NFE2 c.782_785del_1') & (merged_long['gapfill'] == '') & (merged_long['gapfill_start'].notna()),'HGVSc'] = 'NFE2 wildtype'
merged_long.loc[(merged_long['name'] == 'NFE2 c.782_785del_0') & (merged_long['gapfill'] == '') & (merged_long['gapfill_start'].isna()),'name'] = 'NFE2 c.782_785del'
merged_long.loc[(merged_long['name'] == 'NFE2 c.782_785del_1') & (merged_long['gapfill'] == '') & (merged_long['gapfill_start'].notna()),'name'] = 'NFE2 c.782_785del'
##

## Add zero-count rows for every possible HGVSc per probe pair per sample
## Note that this is also handled when computing likelihoods but helpful to have in table for completeness and to make sure we have all combos when plotting
probe_pair_cols = ['lhs_probe', 'rhs_probe']

all_hgvsc_per_probe = (
    merged_long[probe_pair_cols + ['HGVSc']]
    .drop_duplicates()
    .dropna(subset=['HGVSc'])
)

all_sample_probe = (
    merged_long[['sample'] + probe_pair_cols]
    .drop_duplicates()
)

all_combos = all_sample_probe.merge(all_hgvsc_per_probe, on=probe_pair_cols, how='left')

existing = merged_long[['sample'] + probe_pair_cols + ['HGVSc']].drop_duplicates()
missing = all_combos.merge(
    existing,
    on=['sample'] + probe_pair_cols + ['HGVSc'],
    how='left',
    indicator=True
).query('_merge == "left_only"').drop(columns='_merge')

if len(missing) > 0:
    probe_counts = (
        merged_long[['sample'] + probe_pair_cols + ['count_of_this_probe']]
        .drop_duplicates(subset=['sample'] + probe_pair_cols)
    )
    missing = missing.merge(probe_counts, on=['sample'] + probe_pair_cols, how='left')
    missing['count_of_this_gapfill'] = 0

    ref_cols = [c for c in merged_long.columns
                if c not in ['sample', 'count_of_this_gapfill', 'count_of_this_probe','frequency']]
    ref = (
        merged_long[ref_cols]
        .drop_duplicates(subset=probe_pair_cols + ['HGVSc'])
    )

    missing = missing.merge(ref, on=probe_pair_cols + ['HGVSc'], how='left')

    merged_long = pd.concat([merged_long, missing], ignore_index=True)
    print(f'Added {len(missing)} zero-count rows for missing (sample, probe_pair, HGVSc) combos')

print('len merged long', len(merged_long))

merged_long['frequency'] = merged_long['count_of_this_gapfill'] / merged_long['count_of_this_probe']

samples to parse: Index(['BC007_1', 'BC008_1', 'BC005_1', 'BC004_1', 'BC001_1', 'BC010_1',
       'BC012_1', 'BC015_1', 'BC009_1', 'BC014_1', 'BC006_1', 'BC011_1',
       'BC003_1', 'BC016_1', 'BC013_1', 'BC002_1', 'BC012_2', 'BC011_2',
       'BC004_2', 'BC015_2', 'BC003_2', 'BC002_2', 'BC010_2', 'BC001_2',
       'BC008_2', 'BC014_2', 'BC005_2', 'BC006_2', 'BC009_2', 'BC007_2',
       'BC013_2', 'BC016_2', 'BC003_4plex', 'BC002_4plex', 'BC004_4plex',
       'BC001_4plex', 'BC001_1plex'],
      dtype='object')
len merged long 42119
len merged long after merge 42119
Added 162954 zero-count rows for missing (sample, probe_pair, HGVSc) combos
len merged long 205073


In [6]:
### manually add splice variants

merged_long.loc[(merged_long['name'] == 'TP53 c.920-1G>A'),'gapfill_from_transcriptome'] = 'GCT'
merged_long.loc[(merged_long['name'] == 'TP53 c.920-1G>A') & (merged_long['gapfill'] == 'GCT'),'HGVSc'] = 'TP53 wildtype'
merged_long.loc[(merged_long['name'] == 'TP53 c.920-1G>A') & (merged_long['gapfill'] == 'GTT'),'HGVSc'] = 'TP53 c.920-1G>A'

merged_long.loc[(merged_long['name'] == 'TET2 c.4537+2T>G'),'gapfill_from_transcriptome'] = 'TTACC'
merged_long.loc[(merged_long['name'] == 'TET2 c.4537+2T>G') & (merged_long['gapfill'] == 'TTACC'),'HGVSc'] = 'TET2 wildtype'
merged_long.loc[(merged_long['name'] == 'TET2 c.4537+2T>G') & (merged_long['gapfill'] == 'TAACC'),'HGVSc'] = 'TET2 c.4537+3A>T'
merged_long.loc[(merged_long['name'] == 'TET2 c.4537+2T>G') & (merged_long['gapfill'] == 'TTCCC'),'HGVSc'] = 'TET2 c.4537+2T>G'

merged_long.loc[(merged_long['name'] == 'TP53 c.376-2A>G'),'gapfill_from_transcriptome'] = 'CTG'
merged_long.loc[(merged_long['name'] == 'TP53 c.376-2A>G') & (merged_long['gapfill'] == 'CTG'),'HGVSc'] = 'TP53 wildtype'
merged_long.loc[(merged_long['name'] == 'TP53 c.376-2A>G') & (merged_long['gapfill'] == 'CCG'),'HGVSc'] = 'TP53 c.376-2A>G'

merged_long.loc[(merged_long['name'] == 'NFE2 c.115-1G>A'),'gapfill_from_transcriptome'] = 'CCT'
merged_long.loc[(merged_long['name'] == 'NFE2 c.115-1G>A') & (merged_long['gapfill'] == 'CCT'),'HGVSc'] = 'NFE2 wildtype'
merged_long.loc[(merged_long['name'] == 'NFE2 c.115-1G>A') & (merged_long['gapfill'] == 'CTT'),'HGVSc'] = 'NFE2 c.115-1G>A'


In [7]:
### get likelihood of observed gapfills based on different models - edit distance, control sample, or all other samples
merged_long = gf_utils.get_likelihood_given_edit_distance(merged_long)
merged_long = gf_utils.get_likelihood_given_control_sample(merged_long)
merged_long['gapfill_start'] = merged_long['gapfill_start'].fillna('not_found')
merged_long = gf_utils.get_likelihood_given_all_other_samples(merged_long)

merged_long['signed_log_likelihood_given_wt_control'] = np.log10(merged_long['likelihood_given_wt_control'] + (1e-300)) * merged_long['proportion_greater_than_expected_from_control']
merged_long['signed_log_likelihood_given_other_samples'] = np.log10(merged_long['likelihood_given_other_samples'] + (1e-300)) * merged_long['proportion_greater_than_expected_from_others']
merged_long['signed_log_likelihood_given_wt_edit_dist'] = np.log10(merged_long['likelihood_observed_proportion_given_edit_dist'] + (1e-300)) * merged_long['proportion_greater_than_expected_from_edit_dist']

In [9]:
# Create a mapping dictionary from expected_mutations
expected_freq_mapping = expected_mutations.set_index(['sample', 'name'])['expected_frequency_from_bulk'].to_dict()

# Apply the mapping to create the new column
merged_long['expected_frequency_from_bulk'] = merged_long[['sample', 'HGVSc']].apply(
    lambda row: expected_freq_mapping.get((row['sample'], row['HGVSc']), np.nan), axis=1
)

In [10]:
merged_long.to_csv('../output/MPN_all_gapfill_df.csv')

mutated_df = merged_long.loc[~(merged_long['HGVSc'].fillna('').str.contains('wildtype'))]


In [11]:
min_count = 0
min_freq = 0
min_log_likelihood_expected = -3
prior = 25 + min_log_likelihood_expected ### to keep a likelihood of -25 on unexpected
min_log_likelihood = -prior + min_log_likelihood_expected

temp_mutated_df = mutated_df.copy()

### add a prior on expected mutations
for col in ['signed_log_likelihood_given_wt_control', 'signed_log_likelihood_given_other_samples', 'signed_log_likelihood_given_wt_edit_dist']:
    temp_mutated_df.loc[(temp_mutated_df['expected_frequency_from_bulk'].notna()) & (temp_mutated_df[col].notna()), col] -= prior


### tiered approach to define final set
feature_sets = {}
for sample in mutated_df['sample'].unique():
    temp = gf_utils.get_feature_set(temp_mutated_df, sample, likelihood_column = 'signed_log_likelihood_given_wt_control', min_log_likelihood=min_log_likelihood, min_count=min_count, min_frequency=min_freq)
    temp['origin'] = 'from_control'
    feature_set = temp.copy()
    temp = gf_utils.get_feature_set(temp_mutated_df.loc[temp_mutated_df['signed_log_likelihood_given_wt_control'].isna()], sample, likelihood_column = 'signed_log_likelihood_given_other_samples', min_log_likelihood=min_log_likelihood, min_count=min_count, min_frequency=min_freq)
    temp['origin'] = 'from_others'
    feature_set = pd.concat([feature_set, temp], ignore_index=True)
    temp = gf_utils.get_feature_set(temp_mutated_df.loc[(temp_mutated_df['signed_log_likelihood_given_wt_control'].isna()) & (temp_mutated_df['signed_log_likelihood_given_other_samples'].isna())], sample, likelihood_column = 'signed_log_likelihood_given_wt_edit_dist', min_log_likelihood=min_log_likelihood, min_count=min_count, min_frequency=min_freq)
    temp['origin'] = 'from_edit_dist'
    feature_set = pd.concat([feature_set, temp], ignore_index=True)
    feature_sets[sample] = feature_set

all_feature_sets = pd.concat(feature_sets.values(), ignore_index=True)

for col in ['signed_log_likelihood_given_wt_control', 'signed_log_likelihood_given_other_samples', 'signed_log_likelihood_given_wt_edit_dist']:
    all_feature_sets.loc[all_feature_sets['expected_frequency_from_bulk'].notna(), col] += prior

### make feature set to save using frequency and count thresholds only for the mutations that were not expected
all_features_to_save = all_feature_sets.loc[(all_feature_sets['expected_frequency_from_bulk'].notna()) | ((all_feature_sets['count_of_this_gapfill'] > 50) & (all_feature_sets['frequency'] > 0.003))]


In [12]:
### remove loci that frequently appear as novel positives
n_positives = all_features_to_save.loc[all_features_to_save['expected_frequency_from_bulk'].isna()]['name'].value_counts()
to_keep = n_positives[n_positives == 1].index
all_features_to_save = all_features_to_save.loc[(all_features_to_save['expected_frequency_from_bulk'].notna()) | (all_features_to_save['name'].isin(to_keep))]

all_features_to_save['expected_frequency_from_bulk'].isna().value_counts()

expected_frequency_from_bulk
False    119
True       4
Name: count, dtype: int64

In [13]:
### filter further in regime of precision > 0.6
all_features_to_save = all_features_to_save.loc[(all_features_to_save['expected_frequency_from_bulk'].notna()) | ((all_features_to_save['signed_log_likelihood_given_wt_control'] < (-25)) & (all_features_to_save['signed_log_likelihood_given_other_samples'] < (-50)))].sort_values('frequency')

In [14]:
unexpected_positives = all_features_to_save.loc[all_features_to_save['expected_frequency_from_bulk'].isna()]
expected_positives = mutated_df[['name','HGVSc','sample','frequency','expected_frequency_from_bulk']].dropna()

unexpected_positives = unexpected_positives[['name','HGVSc','sample','frequency']].merge(expected_positives[['name','HGVSc','sample','frequency']], on = ['name','sample'], how='left')
to_remove = unexpected_positives.loc[unexpected_positives['frequency_y'].notna()]

### remove unexpected positives that have a similar mutation in the same sample that was expected (these are likely false positives that arise from the same underlying mutation)
all_features_to_save = all_features_to_save.loc[
    ~all_features_to_save[['name', 'sample']].apply(tuple, axis=1).isin(
        to_remove[['name', 'sample']].apply(tuple, axis=1)
    )
]


In [16]:
all_features_to_save.loc[all_features_to_save['expected_frequency_from_bulk'].isna()].sort_values('frequency')

,name,HGVSc,gapfill,gapfill_from_transcriptome,frequency,expected_frequency_from_bulk,lhs_probe,rhs_probe,count_of_this_gapfill,count_of_this_probe,signed_log_likelihood_given_wt_control,signed_log_likelihood_given_other_samples,signed_log_likelihood_given_wt_edit_dist,sample,likelihood_given_wt_edit_dist,original_name,origin
6,NRAS c.35G>A,NRAS c.35G>A,ATCT,ACCT,0.003174,NaN,TTGTCAGTGCGCTTTTCCCAACACC,GCTCCAACCACCACCAGTTTGTACT,153.0,48206.0,-26.648885,-70.356953,-1.397598,BC007_1,0.002472,NRAS c.35G>A,from_control
72,ASXL1 c.4183C>G,ASXL1 c.4183C>G,CCACT,CCAGT,0.078534,NaN,AAGGGCATCCCTTCCAAGTGACCCA,TCCAGGGGACTATGCCCAGTAGCTT,360.0,4584.0,-300.000000,-300.000000,-75.457902,BC003_1,0.002472,ASXL1 c.4183C>G,from_control


In [17]:
VAF_table = mutated_df[['sample','HGVSc','frequency','count_of_this_gapfill','count_of_this_probe','expected_frequency_from_bulk']].dropna()

VAF_table_with_feature_set = VAF_table.merge(all_features_to_save[['sample','HGVSc']], on = ['sample','HGVSc'], how='left', indicator=True)
VAF_table_with_feature_set['_merge'] = VAF_table_with_feature_set['_merge'].replace({'both': 'in_feature_set', 'left_only': 'not_in_feature_set'})

print(len(mutated_df[['sample','HGVSc','frequency','expected_frequency_from_bulk']].dropna()))
print(len(VAF_table_with_feature_set))
print(len(expected_mutations))

pd.concat([VAF_table_with_feature_set, all_features_to_save.loc[all_features_to_save['expected_frequency_from_bulk'].isna(),['sample','HGVSc','frequency','count_of_this_gapfill','count_of_this_probe','expected_frequency_from_bulk']]]).to_csv('../output/all_MPNs_VAF_vs_expected_VAF.csv', index=False)
### 3 excluded mutations were not included in those panels because of overlaps

138
138
143


/tmp/ipykernel_3345382/530847732.py:4: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  VAF_table_with_feature_set['_merge'] = VAF_table_with_feature_set['_merge'].replace({'both': 'in_feature_set', 'left_only': 'not_in_feature_set'})


In [18]:
mutations_bc007 = mutated_df.loc[(mutated_df['sample'] == 'BC007_1')][['HGVSc','frequency','count_of_this_gapfill','count_of_this_probe','expected_frequency_from_bulk']]
mutations_bc007 = mutations_bc007.loc[mutations_bc007['HGVSc'].notna()]
mutations_bc007 = mutations_bc007.merge(all_features_to_save.loc[all_features_to_save['sample'] == 'BC007_1',['HGVSc']], on = 'HGVSc', how='left', indicator=True)
mutations_bc007['in_feature_set'] = mutations_bc007['_merge'] == 'both'
mutations_bc007.drop(columns=['_merge'], inplace=True)
mutations_bc007.sort_values('frequency', ascending=False).to_csv('../output/BC007_1_all_mutations.csv', index=False)